# 04 - 高级RAG技术 (Advanced RAG Techniques)

本notebook深入介绍高级RAG技术，包括混合检索、查询重写、重排序、多跳推理等。

## 内容概览

### Part 1: 混合检索 (Hybrid Retrieval)
- 1.1 Dense vs Sparse 检索对比
- 1.2 Alpha参数调优
- 1.3 融合方法对比 (RRF vs Weighted)

### Part 2: 查询优化 (Query Optimization)
- 2.1 查询扩展 (Query Expansion)
- 2.2 查询分解 (Query Decomposition)
- 2.3 HyDE (Hypothetical Document Embeddings)

### Part 3: 重排序与过滤 (Reranking & Filtering)
- 3.1 Cross-Encoder重排序
- 3.2 多样性过滤 (MMR)
- 3.3 元数据过滤

### Part 4: 高级RAG模式 (Advanced RAG Patterns)
- 4.1 多跳RAG (Multi-hop RAG)
- 4.2 自适应RAG
- 4.3 性能评估与对比

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import time
from typing import List, Dict, Tuple, Optional, Callable
from dataclasses import dataclass, field

from embeddings import DenseEmbedding, SparseEmbedding, HybridEmbedding, EmbeddingConfig
from vector_store import Document, SimpleVectorStore, VectorStoreConfig
from retriever import DenseRetriever, SparseRetriever, HybridRetriever, RetrieverConfig
from rag_pipeline import RAGConfig, SimpleRAGPipeline, AdvancedRAGPipeline

np.random.seed(42)
print("高级RAG技术实验环境已准备就绪")

---
# Part 1: 混合检索 (Hybrid Retrieval)

混合检索结合了稠密检索(语义相似)和稀疏检索(关键词匹配)的优势。

**核心公式:**
$$score_{hybrid} = \alpha \cdot score_{dense} + (1-\alpha) \cdot score_{sparse}$$

**RRF (Reciprocal Rank Fusion):**
$$RRF(d) = \sum_{r \in R} \frac{1}{k + rank_r(d)}$$

## 1.1 Dense vs Sparse 检索对比

In [ ]:
# 准备测试文档集
test_documents = [
    Document(content="机器学习是人工智能的核心分支，通过数据驱动的方式让计算机自动学习和改进", 
             metadata={"category": "AI", "difficulty": "beginner"}),
    Document(content="深度学习使用多层神经网络，在图像识别、语音识别等领域取得突破性进展",
             metadata={"category": "AI", "difficulty": "intermediate"}),
    Document(content="自然语言处理(NLP)让计算机理解、生成和处理人类语言",
             metadata={"category": "NLP", "difficulty": "intermediate"}),
    Document(content="Transformer架构是现代NLP的基础，使用自注意力机制处理序列数据",
             metadata={"category": "NLP", "difficulty": "advanced"}),
    Document(content="BERT是一种预训练语言模型，通过双向编码器学习上下文表示",
             metadata={"category": "NLP", "difficulty": "advanced"}),
    Document(content="GPT系列模型使用自回归方式生成文本，展现出强大的语言理解能力",
             metadata={"category": "LLM", "difficulty": "advanced"}),
    Document(content="RAG检索增强生成结合了检索系统和生成模型，提高回答的准确性",
             metadata={"category": "RAG", "difficulty": "advanced"}),
    Document(content="向量数据库用于存储和检索高维向量，支持相似性搜索",
             metadata={"category": "Database", "difficulty": "intermediate"}),
]

print(f"测试文档数量: {len(test_documents)}")
for i, doc in enumerate(test_documents):
    print(f"  [{i}] {doc.metadata['category']:10} | {doc.content[:40]}...")

In [ ]:
# 初始化检索器
config = EmbeddingConfig(dimension=64)
dense_emb = DenseEmbedding(config, random_seed=42)
store = SimpleVectorStore()

# Dense检索器
dense_retriever = DenseRetriever(dense_emb, store, RetrieverConfig(top_k=5))
dense_retriever.add_documents(test_documents)

# Sparse检索器 (BM25)
sparse_retriever = SparseRetriever(RetrieverConfig(top_k=5))
sparse_retriever.fit(test_documents)

print("检索器初始化完成")

In [ ]:
# 对比测试
test_queries = [
    "什么是机器学习",           # 关键词明确
    "如何让AI理解人类语言",     # 语义查询
    "Transformer注意力机制",   # 专业术语
    "提高问答准确性的方法",     # 抽象查询
]

def compare_retrievers(query: str, dense_ret, sparse_ret):
    """对比Dense和Sparse检索结果"""
    print(f"\n{'='*60}")
    print(f"查询: {query}")
    print(f"{'='*60}")
    
    # Dense检索
    dense_results = dense_ret.retrieve(query)
    print(f"\n[Dense检索] (语义相似)")
    for r in dense_results[:3]:
        print(f"  {r.score:.4f} | {r.document.content[:50]}...")
    
    # Sparse检索
    sparse_results = sparse_ret.retrieve(query)
    print(f"\n[Sparse检索] (关键词匹配)")
    for r in sparse_results[:3]:
        print(f"  {r.score:.4f} | {r.document.content[:50]}...")

for query in test_queries:
    compare_retrievers(query, dense_retriever, sparse_retriever)

## 1.2 Alpha参数调优

Alpha参数控制Dense和Sparse检索的权重比例:
- `alpha=1.0`: 纯Dense检索
- `alpha=0.0`: 纯Sparse检索
- `alpha=0.5-0.7`: 通常最佳平衡点

In [ ]:
def evaluate_alpha(query: str, alpha_values: List[float], 
                   dense_ret, sparse_ret, ground_truth_idx: int):
    """评估不同alpha值的检索效果"""
    results = []
    
    for alpha in alpha_values:
        hybrid = HybridRetriever(
            dense_retriever=dense_ret,
            sparse_retriever=sparse_ret,
            alpha=alpha,
            fusion_method="weighted"
        )
        
        retrieved = hybrid.retrieve(query)
        
        # 计算MRR (Mean Reciprocal Rank)
        mrr = 0.0
        for i, r in enumerate(retrieved):
            if test_documents.index(r.document) == ground_truth_idx:
                mrr = 1.0 / (i + 1)
                break
        
        results.append({
            'alpha': alpha,
            'mrr': mrr,
            'top1_score': retrieved[0].score if retrieved else 0,
            'top1_content': retrieved[0].document.content[:30] if retrieved else ""
        })
    
    return results

# 测试不同alpha值
alpha_values = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
query = "RAG检索增强生成"
ground_truth = 6  # RAG文档的索引

results = evaluate_alpha(query, alpha_values, dense_retriever, sparse_retriever, ground_truth)

print(f"查询: {query}")
print(f"目标文档索引: {ground_truth}")
print(f"\n{'Alpha':>6} | {'MRR':>6} | {'Top1 Score':>10} | Top1 Content")
print("-" * 60)
for r in results:
    print(f"{r['alpha']:>6.1f} | {r['mrr']:>6.2f} | {r['top1_score']:>10.4f} | {r['top1_content']}...")

In [ ]:
# 可视化Alpha调优结果
def visualize_alpha_tuning(results: List[Dict]):
    """ASCII可视化alpha调优结果"""
    print("\nAlpha vs MRR 可视化:")
    print("MRR")
    print("1.0 |" + "-" * 40)
    
    for r in results:
        bar_len = int(r['mrr'] * 40)
        bar = "█" * bar_len + "░" * (40 - bar_len)
        print(f"    | {bar} α={r['alpha']:.1f}")
    
    print("0.0 |" + "-" * 40)
    print("    +" + "-" * 40 + "> Alpha")

visualize_alpha_tuning(results)

## 1.3 融合方法对比 (RRF vs Weighted)

In [ ]:
def compare_fusion_methods(query: str, dense_ret, sparse_ret):
    """对比RRF和Weighted融合方法"""
    print(f"\n查询: {query}")
    print("=" * 70)
    
    # Weighted融合
    hybrid_weighted = HybridRetriever(
        dense_retriever=dense_ret,
        sparse_retriever=sparse_ret,
        alpha=0.6,
        fusion_method="weighted"
    )
    
    # RRF融合
    hybrid_rrf = HybridRetriever(
        dense_retriever=dense_ret,
        sparse_retriever=sparse_ret,
        alpha=0.6,
        fusion_method="rrf"
    )
    
    weighted_results = hybrid_weighted.retrieve(query)
    rrf_results = hybrid_rrf.retrieve(query)
    
    print(f"\n{'Weighted融合':^35} | {'RRF融合':^35}")
    print("-" * 70)
    
    for i in range(min(3, len(weighted_results), len(rrf_results))):
        w = weighted_results[i]
        r = rrf_results[i]
        print(f"{w.score:.4f} {w.document.content[:25]:25} | {r.score:.4f} {r.document.content[:25]}")

for query in ["机器学习基础", "NLP语言模型", "向量检索"]:
    compare_fusion_methods(query, dense_retriever, sparse_retriever)

---
# Part 2: 查询优化 (Query Optimization)

查询优化通过改进用户查询来提高检索效果。

## 2.1 查询扩展 (Query Expansion)

通过添加同义词、相关词来扩展原始查询。

In [ ]:
class QueryExpander:
    """查询扩展器"""
    
    def __init__(self):
        # 同义词词典
        self.synonyms = {
            "机器学习": ["ML", "machine learning", "自动学习"],
            "深度学习": ["DL", "deep learning", "神经网络学习"],
            "自然语言处理": ["NLP", "文本处理", "语言理解"],
            "人工智能": ["AI", "智能系统", "智能算法"],
            "向量": ["embedding", "嵌入", "表示向量"],
            "检索": ["搜索", "查找", "retrieval"],
        }
    
    def expand(self, query: str, max_expansions: int = 3) -> List[str]:
        """扩展查询"""
        expanded = [query]
        
        for term, syns in self.synonyms.items():
            if term in query:
                for syn in syns[:max_expansions]:
                    expanded.append(query.replace(term, syn))
        
        return expanded[:max_expansions + 1]

expander = QueryExpander()

test_queries = ["机器学习基础", "自然语言处理技术", "向量检索方法"]
for q in test_queries:
    expanded = expander.expand(q)
    print(f"\n原始查询: {q}")
    print(f"扩展查询: {expanded}")

In [ ]:
def multi_query_retrieve(queries: List[str], retriever, top_k: int = 5) -> List:
    """多查询检索并融合结果"""
    all_results = {}
    
    for query in queries:
        results = retriever.retrieve(query)
        for r in results:
            doc_id = id(r.document)
            if doc_id not in all_results:
                all_results[doc_id] = {'doc': r.document, 'scores': [], 'count': 0}
            all_results[doc_id]['scores'].append(r.score)
            all_results[doc_id]['count'] += 1
    
    # 按平均分数排序
    ranked = sorted(
        all_results.values(),
        key=lambda x: sum(x['scores']) / len(x['scores']) * x['count'],
        reverse=True
    )
    
    return ranked[:top_k]

# 测试多查询检索
query = "机器学习"
expanded_queries = expander.expand(query)

print(f"扩展查询: {expanded_queries}")
results = multi_query_retrieve(expanded_queries, dense_retriever)

print(f"\n多查询检索结果:")
for i, r in enumerate(results):
    avg_score = sum(r['scores']) / len(r['scores'])
    print(f"  [{i+1}] 平均分:{avg_score:.4f} 命中:{r['count']}次 | {r['doc'].content[:40]}...")

## 2.2 查询分解 (Query Decomposition)

将复杂查询分解为多个简单子查询。

In [ ]:
class QueryDecomposer:
    """查询分解器"""
    
    def __init__(self):
        self.decomposition_patterns = [
            ("和", "与", "以及"),  # 并列关系
            ("的区别", "的不同", "vs"),  # 对比关系
            ("如何", "怎么", "怎样"),  # 方法类
        ]
    
    def decompose(self, query: str) -> List[str]:
        """分解复杂查询"""
        sub_queries = []
        
        # 检测并列关系
        for conj in ["和", "与", "以及", ",", "，"]:
            if conj in query:
                parts = query.split(conj)
                sub_queries.extend([p.strip() for p in parts if p.strip()])
                break
        
        # 检测对比关系
        if "的区别" in query or "的不同" in query:
            base = query.replace("的区别", "").replace("的不同", "")
            for conj in ["和", "与"]:
                if conj in base:
                    parts = base.split(conj)
                    sub_queries = [f"什么是{p.strip()}" for p in parts]
                    sub_queries.append(query)  # 保留原始对比查询
                    break
        
        return sub_queries if sub_queries else [query]

decomposer = QueryDecomposer()

complex_queries = [
    "机器学习和深度学习的区别",
    "BERT与GPT的不同",
    "NLP、计算机视觉以及语音识别",
]

for q in complex_queries:
    sub_queries = decomposer.decompose(q)
    print(f"\n原始查询: {q}")
    print(f"分解结果: {sub_queries}")

## 2.3 HyDE (Hypothetical Document Embeddings)

生成假设性文档，用其embedding进行检索。

**原理:** 先让LLM生成一个假设的答案文档，然后用这个文档的embedding去检索真实文档。

In [ ]:
class HyDERetriever:
    """HyDE检索器 (简化版)"""
    
    def __init__(self, base_retriever, hypothetical_generator: Callable[[str], str]):
        self.base_retriever = base_retriever
        self.generator = hypothetical_generator
    
    def retrieve(self, query: str, top_k: int = 5):
        """使用HyDE进行检索"""
        # 生成假设性文档
        hypothetical_doc = self.generator(query)
        
        # 使用假设文档进行检索
        results = self.base_retriever.retrieve(hypothetical_doc)
        
        return results[:top_k], hypothetical_doc

# 简单的假设文档生成器 (实际应用中使用LLM)
def simple_hypothetical_generator(query: str) -> str:
    """简单假设文档生成器"""
    templates = {
        "什么是": "{topic}是一种重要的技术，它通过特定的方法和算法来实现目标。",
        "如何": "要{action}，需要遵循以下步骤：首先理解基本概念，然后选择合适的方法。",
        "default": "{query}是一个重要的概念，在人工智能领域有广泛应用。"
    }
    
    if "什么是" in query:
        topic = query.replace("什么是", "").strip()
        return templates["什么是"].format(topic=topic)
    elif "如何" in query:
        action = query.replace("如何", "").strip()
        return templates["如何"].format(action=action)
    else:
        return templates["default"].format(query=query)

# 测试HyDE
hyde_retriever = HyDERetriever(dense_retriever, simple_hypothetical_generator)

test_queries = ["什么是机器学习", "如何理解自然语言", "RAG技术"]

for query in test_queries:
    results, hypo_doc = hyde_retriever.retrieve(query, top_k=3)
    print(f"\n查询: {query}")
    print(f"假设文档: {hypo_doc}")
    print(f"检索结果:")
    for r in results:
        print(f"  {r.score:.4f} | {r.document.content[:50]}...")

---
# Part 3: 重排序与过滤 (Reranking & Filtering)

重排序和过滤用于优化初始检索结果。

## 3.1 Cross-Encoder重排序

Cross-Encoder同时编码query和document，提供更精确的相关性评分。

In [ ]:
class SimpleReranker:
    """简单重排序器 (模拟Cross-Encoder)"""
    
    def __init__(self, keyword_boost: float = 0.3):
        self.keyword_boost = keyword_boost
    
    def rerank(self, query: str, results: List, top_k: int = 5) -> List:
        """重排序检索结果"""
        query_terms = set(query.lower())
        
        reranked = []
        for r in results:
            doc_content = r.document.content.lower()
            
            # 计算关键词重叠度
            overlap = sum(1 for term in query_terms if term in doc_content)
            keyword_score = overlap / max(len(query_terms), 1)
            
            # 综合分数
            new_score = r.score * (1 - self.keyword_boost) + keyword_score * self.keyword_boost
            
            reranked.append({
                'document': r.document,
                'original_score': r.score,
                'reranked_score': new_score,
                'keyword_score': keyword_score
            })
        
        reranked.sort(key=lambda x: x['reranked_score'], reverse=True)
        return reranked[:top_k]

# 测试重排序
reranker = SimpleReranker(keyword_boost=0.4)
query = "Transformer自注意力"

# 获取初始结果
initial_results = dense_retriever.retrieve(query)

print(f"查询: {query}")
print(f"\n初始排序:")
for i, r in enumerate(initial_results[:5]):
    print(f"  [{i+1}] {r.score:.4f} | {r.document.content[:45]}...")

# 重排序
reranked = reranker.rerank(query, initial_results)

print(f"\n重排序后:")
for i, r in enumerate(reranked):
    print(f"  [{i+1}] {r['reranked_score']:.4f} (原:{r['original_score']:.4f}) | {r['document'].content[:35]}...")

## 3.2 多样性过滤 (MMR - Maximal Marginal Relevance)

MMR在保持相关性的同时增加结果多样性。

$$MMR = \arg\max_{d_i \in R \setminus S} [\lambda \cdot Sim(d_i, q) - (1-\lambda) \cdot \max_{d_j \in S} Sim(d_i, d_j)]$$

In [ ]:
class MMRFilter:
    """MMR多样性过滤器"""
    
    def __init__(self, embedding_model, lambda_param: float = 0.7):
        self.embedding = embedding_model
        self.lambda_param = lambda_param
    
    def _cosine_similarity(self, v1: np.ndarray, v2: np.ndarray) -> float:
        """计算余弦相似度"""
        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)
        if norm1 == 0 or norm2 == 0:
            return 0.0
        return np.dot(v1, v2) / (norm1 * norm2)
    
    def filter(self, query: str, results: List, top_k: int = 5) -> List:
        """使用MMR过滤结果"""
        if len(results) <= top_k:
            return results
        
        # 获取embeddings
        query_emb = self.embedding.embed(query)
        doc_embs = [self.embedding.embed(r.document.content) for r in results]
        
        selected = []
        remaining = list(range(len(results)))
        
        while len(selected) < top_k and remaining:
            best_idx = None
            best_score = float('-inf')
            
            for idx in remaining:
                # 与query的相似度
                relevance = self._cosine_similarity(doc_embs[idx], query_emb)
                
                # 与已选文档的最大相似度
                max_sim = 0.0
                for sel_idx in selected:
                    sim = self._cosine_similarity(doc_embs[idx], doc_embs[sel_idx])
                    max_sim = max(max_sim, sim)
                
                # MMR分数
                mmr_score = self.lambda_param * relevance - (1 - self.lambda_param) * max_sim
                
                if mmr_score > best_score:
                    best_score = mmr_score
                    best_idx = idx
            
            if best_idx is not None:
                selected.append(best_idx)
                remaining.remove(best_idx)
        
        return [results[i] for i in selected]

# 测试MMR
mmr_filter = MMRFilter(dense_emb, lambda_param=0.6)
query = "人工智能技术"

initial_results = dense_retriever.retrieve(query)
mmr_results = mmr_filter.filter(query, initial_results, top_k=4)

print(f"查询: {query}")
print(f"\n原始Top-4:")
for i, r in enumerate(initial_results[:4]):
    print(f"  [{i+1}] {r.document.metadata['category']:10} | {r.document.content[:40]}...")

print(f"\nMMR过滤后 (更多样化):")
for i, r in enumerate(mmr_results):
    print(f"  [{i+1}] {r.document.metadata['category']:10} | {r.document.content[:40]}...")

## 3.3 元数据过滤

In [ ]:
class MetadataFilter:
    """元数据过滤器"""
    
    def filter(self, results: List, filters: Dict) -> List:
        """根据元数据过滤结果"""
        filtered = []
        
        for r in results:
            match = True
            for key, value in filters.items():
                if key not in r.document.metadata:
                    match = False
                    break
                
                doc_value = r.document.metadata[key]
                
                # 支持列表匹配
                if isinstance(value, list):
                    if doc_value not in value:
                        match = False
                        break
                else:
                    if doc_value != value:
                        match = False
                        break
            
            if match:
                filtered.append(r)
        
        return filtered

# 测试元数据过滤
meta_filter = MetadataFilter()
query = "人工智能"

all_results = dense_retriever.retrieve(query)

print(f"查询: {query}")
print(f"\n所有结果 ({len(all_results)}个):")
for r in all_results:
    print(f"  {r.document.metadata['category']:10} {r.document.metadata['difficulty']:12} | {r.document.content[:35]}...")

# 过滤: 只要NLP类别
nlp_results = meta_filter.filter(all_results, {'category': 'NLP'})
print(f"\n过滤后 (category=NLP, {len(nlp_results)}个):")
for r in nlp_results:
    print(f"  {r.document.metadata['category']:10} | {r.document.content[:45]}...")

# 过滤: 高级难度
advanced_results = meta_filter.filter(all_results, {'difficulty': 'advanced'})
print(f"\n过滤后 (difficulty=advanced, {len(advanced_results)}个):")
for r in advanced_results:
    print(f"  {r.document.metadata['difficulty']:12} | {r.document.content[:45]}...")

---
# Part 4: 高级RAG模式 (Advanced RAG Patterns)

高级RAG模式包括多跳推理、自适应检索等技术。

## 4.1 多跳RAG (Multi-hop RAG)

多跳RAG通过多轮检索来回答需要综合多个信息源的复杂问题。

In [ ]:
class MultiHopRAG:
    """多跳RAG系统"""
    
    def __init__(self, retriever, max_hops: int = 3):
        self.retriever = retriever
        self.max_hops = max_hops
    
    def _extract_entities(self, text: str) -> List[str]:
        """提取文本中的关键实体 (简化版)"""
        keywords = ["机器学习", "深度学习", "NLP", "Transformer", "BERT", "GPT", "RAG", "向量"]
        return [kw for kw in keywords if kw in text]
    
    def _generate_followup_query(self, query: str, context: str) -> Optional[str]:
        """生成后续查询 (简化版)"""
        entities = self._extract_entities(context)
        query_entities = self._extract_entities(query)
        
        # 找到新实体作为后续查询
        new_entities = [e for e in entities if e not in query_entities]
        if new_entities:
            return f"{new_entities[0]}的详细介绍"
        return None
    
    def retrieve(self, query: str) -> Dict:
        """多跳检索"""
        all_docs = []
        queries = [query]
        hop_results = []
        
        current_query = query
        for hop in range(self.max_hops):
            # 检索
            results = self.retriever.retrieve(current_query)
            
            hop_info = {
                'hop': hop + 1,
                'query': current_query,
                'results': results[:2]
            }
            hop_results.append(hop_info)
            
            # 收集文档
            for r in results[:2]:
                if r.document not in [d['doc'] for d in all_docs]:
                    all_docs.append({'doc': r.document, 'hop': hop + 1})
            
            # 生成后续查询
            if results:
                context = results[0].document.content
                followup = self._generate_followup_query(current_query, context)
                if followup and followup not in queries:
                    queries.append(followup)
                    current_query = followup
                else:
                    break
            else:
                break
        
        return {
            'documents': all_docs,
            'hop_results': hop_results,
            'total_hops': len(hop_results)
        }

# 测试多跳RAG
multi_hop = MultiHopRAG(dense_retriever, max_hops=3)
query = "什么是机器学习"

result = multi_hop.retrieve(query)

print(f"初始查询: {query}")
print(f"总跳数: {result['total_hops']}")
print(f"\n检索过程:")
for hop in result['hop_results']:
    print(f"\n  [Hop {hop['hop']}] 查询: {hop['query']}")
    for r in hop['results']:
        print(f"    -> {r.document.content[:50]}...")

print(f"\n收集的文档 ({len(result['documents'])}个):")
for d in result['documents']:
    print(f"  [Hop {d['hop']}] {d['doc'].content[:55]}...")

## 4.2 自适应RAG

根据查询类型自动选择最佳检索策略。

In [ ]:
class AdaptiveRAG:
    """自适应RAG系统"""
    
    def __init__(self, dense_retriever, sparse_retriever):
        self.dense = dense_retriever
        self.sparse = sparse_retriever
        self.hybrid = HybridRetriever(dense_retriever, sparse_retriever, alpha=0.6)
    
    def _classify_query(self, query: str) -> str:
        """分类查询类型"""
        # 关键词查询 (包含专业术语)
        technical_terms = ["BERT", "GPT", "Transformer", "BM25", "RRF", "HNSW"]
        if any(term in query for term in technical_terms):
            return "keyword"
        
        # 语义查询 (抽象问题)
        semantic_patterns = ["如何", "为什么", "什么是", "怎么", "区别"]
        if any(p in query for p in semantic_patterns):
            return "semantic"
        
        # 默认混合
        return "hybrid"
    
    def retrieve(self, query: str, top_k: int = 5) -> Dict:
        """自适应检索"""
        query_type = self._classify_query(query)
        
        if query_type == "keyword":
            retriever = self.sparse
            strategy = "Sparse (BM25)"
        elif query_type == "semantic":
            retriever = self.dense
            strategy = "Dense (Embedding)"
        else:
            retriever = self.hybrid
            strategy = "Hybrid (Dense + Sparse)"
        
        results = retriever.retrieve(query)
        
        return {
            'query_type': query_type,
            'strategy': strategy,
            'results': results[:top_k]
        }

# 测试自适应RAG
adaptive = AdaptiveRAG(dense_retriever, sparse_retriever)

test_queries = [
    "BERT模型",                    # 关键词查询
    "如何理解自然语言",            # 语义查询
    "机器学习技术应用",            # 混合查询
]

for query in test_queries:
    result = adaptive.retrieve(query, top_k=3)
    print(f"\n查询: {query}")
    print(f"类型: {result['query_type']} -> 策略: {result['strategy']}")
    print(f"结果:")
    for r in result['results']:
        print(f"  {r.score:.4f} | {r.document.content[:45]}...")

## 4.3 性能评估与对比

In [ ]:
class RAGEvaluator:
    """RAG性能评估器"""
    
    def __init__(self):
        self.metrics = {}
    
    def evaluate(self, retriever, queries: List[str], 
                 ground_truth: Dict[str, List[int]], name: str) -> Dict:
        """评估检索器性能"""
        total_mrr = 0.0
        total_recall = 0.0
        total_precision = 0.0
        total_time = 0.0
        
        for query in queries:
            if query not in ground_truth:
                continue
            
            relevant_ids = ground_truth[query]
            
            start = time.time()
            results = retriever.retrieve(query)
            elapsed = time.time() - start
            total_time += elapsed
            
            # 计算MRR
            for i, r in enumerate(results):
                doc_idx = test_documents.index(r.document)
                if doc_idx in relevant_ids:
                    total_mrr += 1.0 / (i + 1)
                    break
            
            # 计算Recall@5和Precision@5
            retrieved_ids = [test_documents.index(r.document) for r in results[:5]]
            hits = len(set(retrieved_ids) & set(relevant_ids))
            total_recall += hits / len(relevant_ids) if relevant_ids else 0
            total_precision += hits / 5
        
        n = len(queries)
        metrics = {
            'name': name,
            'MRR': total_mrr / n,
            'Recall@5': total_recall / n,
            'Precision@5': total_precision / n,
            'Avg_Time_ms': (total_time / n) * 1000
        }
        
        self.metrics[name] = metrics
        return metrics
    
    def compare(self) -> None:
        """对比所有评估结果"""
        print(f"\n{'检索器':<20} | {'MRR':>8} | {'Recall@5':>10} | {'Precision@5':>12} | {'Time(ms)':>10}")
        print("-" * 75)
        for name, m in self.metrics.items():
            print(f"{name:<20} | {m['MRR']:>8.4f} | {m['Recall@5']:>10.4f} | {m['Precision@5']:>12.4f} | {m['Avg_Time_ms']:>10.2f}")

# 准备评估数据
eval_queries = ["机器学习", "自然语言处理", "RAG检索"]
ground_truth = {
    "机器学习": [0, 1],      # 机器学习、深度学习文档
    "自然语言处理": [2, 3, 4],  # NLP相关文档
    "RAG检索": [6, 7],       # RAG、向量数据库文档
}

# 评估不同检索器
evaluator = RAGEvaluator()

evaluator.evaluate(dense_retriever, eval_queries, ground_truth, "Dense")
evaluator.evaluate(sparse_retriever, eval_queries, ground_truth, "Sparse (BM25)")

hybrid_06 = HybridRetriever(dense_retriever, sparse_retriever, alpha=0.6)
evaluator.evaluate(hybrid_06, eval_queries, ground_truth, "Hybrid (α=0.6)")

hybrid_08 = HybridRetriever(dense_retriever, sparse_retriever, alpha=0.8)
evaluator.evaluate(hybrid_08, eval_queries, ground_truth, "Hybrid (α=0.8)")

evaluator.compare()

In [ ]:
# 可视化性能对比
def visualize_comparison(metrics: Dict):
    """ASCII可视化性能对比"""
    print("\n性能对比可视化 (MRR):")
    print("=" * 50)
    
    max_mrr = max(m['MRR'] for m in metrics.values())
    
    for name, m in metrics.items():
        bar_len = int((m['MRR'] / max_mrr) * 30) if max_mrr > 0 else 0
        bar = "█" * bar_len + "░" * (30 - bar_len)
        print(f"{name:<18} |{bar}| {m['MRR']:.4f}")
    
    print("\n性能对比可视化 (Recall@5):")
    print("=" * 50)
    
    max_recall = max(m['Recall@5'] for m in metrics.values())
    
    for name, m in metrics.items():
        bar_len = int((m['Recall@5'] / max_recall) * 30) if max_recall > 0 else 0
        bar = "█" * bar_len + "░" * (30 - bar_len)
        print(f"{name:<18} |{bar}| {m['Recall@5']:.4f}")

visualize_comparison(evaluator.metrics)

## 总结

本notebook介绍了以下高级RAG技术:

1. **混合检索**: 结合Dense和Sparse检索的优势
2. **查询优化**: 查询扩展、分解、HyDE
3. **重排序与过滤**: Cross-Encoder、MMR、元数据过滤
4. **高级模式**: 多跳RAG、自适应RAG

关键要点:
- Alpha参数调优对混合检索效果影响显著
- 查询优化可以显著提升检索召回率
- MMR可以增加结果多样性
- 自适应策略可以根据查询类型选择最佳方法